
# NeoLM — Perplexity Benchmark

Evaluate the released [NeoLM-Small-Base](https://huggingface.co/MaagDeveloper/NeoLM-Small-Base) on the held-out NeoCorpus test split.

## Evaluation

- **Model:** NeoLM-Small-Base (61.36M parameters)
- **Dataset:** NeoCorpus — Test Split
- **Test tokens:** 12,094,582
- **Context length:** 1,024
- **Stride:** 1,023

## Metrics

- Cross-Entropy Loss
- Perplexity (PPL)
- Evaluation Runtime
- Tokens per Second
- Peak GPU Memory

The model is evaluated without training or updating its weights.

**Note:** Perplexity values are tokenizer-dependent and should not be directly compared across models with different tokenizers.


In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM
import math
from tqdm.auto import tqdm

ROOT = Path("..").resolve()

MODEL_DIR = ROOT / "artifacts/model"
TEST_FILE = ROOT / "artifacts/pretrain_db/test.bin"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load released model
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    local_files_only=True,
).to(device).eval()

# Dataset configuration
BLOCK_SIZE = model.config.max_position_embeddings
STRIDE = BLOCK_SIZE - 1

# Memory-mapped test tokens
test_tokens = np.memmap(
    TEST_FILE,
    dtype=np.uint16,
    mode="r",
)

num_blocks = (
    (len(test_tokens) - BLOCK_SIZE) // STRIDE + 1
)

print("=" * 60)
print("NEOLM — PERPLEXITY BENCHMARK")
print("=" * 60)
print(f"Model parameters: {model.num_parameters():,}")
print(f"Device:           {device}")
print(f"Precision:        {next(model.parameters()).dtype}")
print(f"Test tokens:      {len(test_tokens):,}")
print(f"Context length:   {BLOCK_SIZE:,}")
print(f"Stride:           {STRIDE:,}")
print(f"Test blocks:      {num_blocks:,}")
print("=" * 60)

In [ ]:
BATCH_SIZE = 2

total_nll = 0.0
total_predictions = 0

if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.inference_mode():
    for i in tqdm(
        range(0, num_blocks, BATCH_SIZE),
        desc="Evaluating NeoLM",
        unit="batch",
    ):
        end = min(i + BATCH_SIZE, num_blocks)

        batch = np.stack([
            test_tokens[j * STRIDE:j * STRIDE + BLOCK_SIZE]
            for j in range(i, end)
        ]).astype(np.int64)

        input_ids = torch.from_numpy(batch).to(device)

        outputs = model(
            input_ids=input_ids,
            labels=input_ids,
            use_cache=False,
        )

        # Mean loss -> summed negative log-likelihood
        predictions = input_ids.numel() - input_ids.size(0)

        total_nll += outputs.loss.item() * predictions
        total_predictions += predictions

        del outputs, input_ids

    # Include the remaining tokens after the last full block
    last_end = (num_blocks - 1) * STRIDE + BLOCK_SIZE

    if last_end < len(test_tokens):
        tail = np.asarray(
            test_tokens[last_end - 1:],
            dtype=np.int64,
        )

        input_ids = torch.from_numpy(
            tail.copy()
        ).unsqueeze(0).to(device)

        outputs = model(
            input_ids=input_ids,
            labels=input_ids,
            use_cache=False,
        )

        predictions = input_ids.size(1) - 1
        total_nll += outputs.loss.item() * predictions
        total_predictions += predictions

if device == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

test_loss = total_nll / total_predictions
test_ppl = math.exp(test_loss)

tokens_per_second = total_predictions / elapsed

peak_vram = (
    torch.cuda.max_memory_allocated() / 1024**3
    if device == "cuda"
    else 0.0
)

assert total_predictions == len(test_tokens) - 1

print(f"\n{'=' * 60}")
print("NEOLM — FULL TEST EVALUATION")
print(f"{'=' * 60}")
print(f"Test tokens:       {len(test_tokens):,}")
print(f"Predicted tokens:  {total_predictions:,}")
print(f"Test loss:         {test_loss:.4f}")
print(f"Test perplexity:   {test_ppl:.4f}")
print(f"Runtime:           {elapsed / 60:.2f} minutes")
print(f"Tokens/sec:        {tokens_per_second:,.0f}")
print(f"Peak GPU memory:   {peak_vram:.2f} GiB")
print(f"{'=' * 60}")

In [ ]:
from datasets import load_from_disk

test_ds = load_from_disk(
    str(ROOT / "artifacts/pretrain_db/final")
)["test"]

total_bytes = sum(
    len(row["text"].strip().encode("utf-8"))
    for row in test_ds
)

bits_per_byte = total_nll / (total_bytes * math.log(2))

print("=" * 60)
print("NEOLM — BITS PER BYTE")
print("=" * 60)
print(f"Test bytes:       {total_bytes:,}")
print(f"Test tokens:      {len(test_tokens):,}")
print(f"Bytes/token:      {total_bytes / len(test_tokens):.4f}")
print(f"Bits per byte:    {bits_per_byte:.4f}")
print("=" * 60)

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

results = {
    "model": "MaagDeveloper/NeoLM-Small-Base",
    "benchmark": "perplexity",
    "dataset": "MaagDeveloper/NeoCorpus",
    "split": "test",
    "precision": str(next(model.parameters()).dtype),
    "test_tokens": len(test_tokens),
    "predicted_tokens": total_predictions,
    "test_loss": test_loss,
    "perplexity": test_ppl,
    "bits_per_byte": bits_per_byte,
    "runtime_seconds": elapsed,
    "tokens_per_second": tokens_per_second,
    "peak_vram_gib": peak_vram,
}

output_file = RESULTS_DIR / "neolm_perplexity.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print("NEOLM — PERPLEXITY RESULTS SAVED")
print("=" * 60)
print(f"Perplexity:    {test_ppl:.4f}")
print(f"Bits per Byte: {bits_per_byte:.4f}")
print(f"Saved to:      {output_file}")
print("=" * 60)